This file is for testing various regularization techniques and will be using that same targeted class 35 and 36 augmentation 

I will be using DropOut, Weight Decay Training, Stochastic Depth

In [1]:
import pandas as pd
import regex as re
import torch

In [2]:
df = pd.read_csv(r'D:\Traffic\labels_processed.csv')

In [3]:
def label_function(dpath):
    class_name = re.findall(r'(\d+)_.*\.png$', dpath.name)
    class_id = int(class_name[0])
    return class_id

In [4]:
from pathlib import Path

In [5]:
path = Path(r'D:\Traffic\traffic_Data_processed\DATA')

In [6]:
lr_head = 0.0017378008365631102
lr_whole_model = 1.9054607491852948e-06

In [7]:
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [8]:
from torch.utils.data import Dataset
from PIL import Image

In [9]:
class dset(Dataset):
    def __init__(self, image_paths, transform = None):
        self.image_paths = image_paths
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image_path = self.image_paths[idx]
        image = Image.open(image_path).convert("RGB")
        label = label_function(Path(image_path))
        if self.transform:
            image = self.transform(image)
        return image, label

In [10]:
image_paths = list(path.rglob("*.png"))

In [11]:
from torchvision import transforms
from torch.utils.data import random_split
from torch.utils.data import DataLoader

In [12]:
import torchvision
num_classes = 55
device = torch.device("cuda")

In [13]:
import kornia.augmentation as K
import torch.nn as nn

First i am using p = 0.2 for dropout and then 0.3 and 0.5

In [15]:
transform_1 = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

dataset_1 = dset(image_paths = image_paths, transform = transform_1)

ts = int(0.75*len(dataset_1))
vs = len(dataset_1) - ts
generator = torch.Generator().manual_seed(42)
train_dataset, valid_dataset = random_split(dataset_1, [ts, vs], generator)

train_loader_1 = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True
)

valid_loader_1 = DataLoader(
    valid_dataset,
    batch_size=16,
    shuffle=False
)

model_1 = torchvision.models.resnet34(weights="DEFAULT")

model_1.fc = nn.Sequential(
    nn.Dropout(p=0.2),
    nn.Linear(
        model_1.fc.in_features,
        num_classes
    )
)

model_1 = model_1.to(device)

for param in model_1.parameters():
    param.requires_grad = False

for param in model_1.fc.parameters():
    param.requires_grad = True

train_aug_1 = K.AugmentationSequential(
    K.ColorJiggle(
        contrast = 0.2,
        p = 0.5
    ),
    K.RandomPlanckianJitter(
        mode = "CIED",
        p = 0.5
    )
).to(device)

targeted_aug_1 = K.AugmentationSequential(
    K.RandomRotation(
        degrees = 10,
        p = 0.5
    ),
    K.RandomAffine(
        degrees = 0,
        scale = (0.9, 1.1),
        p = 0.5
    ),
    K.RandomPerspective(
        distortion_scale = 0.2,
        p = 0.5
    ),
    K.ColorJiggle(
        brightness = 0.2,
        contrast = 0.2,
        p = 0.5
    )
).to(device)

optimizer_1 = torch.optim.RMSprop(
    model_1.fc.parameters(),
    lr=lr_head,
    weight_decay=1e-3
)

scheduler_1 = torch.optim.lr_scheduler.OneCycleLR(
    optimizer_1,
    max_lr = lr_head,
    epochs = 20,
    steps_per_epoch = len(train_loader_1)
)

criterion = nn.CrossEntropyLoss()

for epoch in range(20):
    model_1.train()
    train_loss = 0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader_1:

        images = images.to(device)
        labels = labels.to(device)

        images = train_aug_1(images)

        mask = (labels == 35) | (labels == 36)

        if mask.any():
            images[mask] = targeted_aug_1(images[mask])

        optimizer_1.zero_grad()

        outputs = model_1(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer_1.step()

        scheduler_1.step()

        train_loss += loss.item()

        _, predicted = outputs.max(1)

        train_correct += (predicted == labels).sum().item()

        train_total += labels.size(0)

    train_acc = 100 * train_correct / train_total

    model_1.eval()

    valid_loss = 0
    valid_correct = 0
    valid_total = 0

    with torch.no_grad():

        for images, labels in valid_loader_1:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model_1(images)

            loss = criterion(outputs, labels)

            valid_loss += loss.item()

            _, predicted = outputs.max(1)

            valid_correct += (predicted == labels).sum().item()

            valid_total += labels.size(0)

    valid_acc = 100 * valid_correct / valid_total

    print(
        f"Epoch {epoch+1}/20 | "
        f"Train Acc: {train_acc:.2f}% | "
        f"Valid Acc: {valid_acc:.2f}%"
    )

Epoch 1/20 | Train Acc: 61.68% | Valid Acc: 86.13%
Epoch 2/20 | Train Acc: 83.81% | Valid Acc: 88.73%
Epoch 3/20 | Train Acc: 84.23% | Valid Acc: 88.82%
Epoch 4/20 | Train Acc: 82.97% | Valid Acc: 89.11%
Epoch 5/20 | Train Acc: 83.78% | Valid Acc: 91.62%
Epoch 6/20 | Train Acc: 85.00% | Valid Acc: 89.79%
Epoch 7/20 | Train Acc: 86.54% | Valid Acc: 91.04%
Epoch 8/20 | Train Acc: 87.89% | Valid Acc: 91.04%
Epoch 9/20 | Train Acc: 87.57% | Valid Acc: 94.70%
Epoch 10/20 | Train Acc: 87.34% | Valid Acc: 91.62%
Epoch 11/20 | Train Acc: 88.63% | Valid Acc: 95.47%
Epoch 12/20 | Train Acc: 88.44% | Valid Acc: 94.99%
Epoch 13/20 | Train Acc: 89.88% | Valid Acc: 95.95%
Epoch 14/20 | Train Acc: 89.72% | Valid Acc: 95.18%
Epoch 15/20 | Train Acc: 92.26% | Valid Acc: 96.44%
Epoch 16/20 | Train Acc: 93.00% | Valid Acc: 95.95%
Epoch 17/20 | Train Acc: 93.93% | Valid Acc: 97.11%
Epoch 18/20 | Train Acc: 95.02% | Valid Acc: 97.69%
Epoch 19/20 | Train Acc: 96.72% | Valid Acc: 97.88%
Epoch 20/20 | Train A

In [16]:
transform_2 = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

dataset_2 = dset(image_paths = image_paths, transform = transform_2)

ts = int(0.75*len(dataset_2))
vs = len(dataset_2) - ts
generator = torch.Generator().manual_seed(42)
train_dataset, valid_dataset = random_split(dataset_2, [ts, vs], generator)

train_loader_2 = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True
)

valid_loader_2 = DataLoader(
    valid_dataset,
    batch_size=16,
    shuffle=False
)

model_2 = torchvision.models.resnet34(weights="DEFAULT")

model_2.fc = nn.Sequential(
    nn.Dropout(p=0.3),
    nn.Linear(
        model_2.fc.in_features,
        num_classes
    )
)

model_2 = model_2.to(device)

for param in model_2.parameters():
    param.requires_grad = False

for param in model_2.fc.parameters():
    param.requires_grad = True

train_aug_2 = K.AugmentationSequential(
    K.ColorJiggle(
        contrast = 0.2,
        p = 0.5
    ),
    K.RandomPlanckianJitter(
        mode = "CIED",
        p = 0.5
    )
).to(device)

targeted_aug_2 = K.AugmentationSequential(
    K.RandomRotation(
        degrees = 10,
        p = 0.5
    ),
    K.RandomAffine(
        degrees = 0,
        scale = (0.9, 1.1),
        p = 0.5
    ),
    K.RandomPerspective(
        distortion_scale = 0.2,
        p = 0.5
    ),
    K.ColorJiggle(
        brightness = 0.2,
        contrast = 0.2,
        p = 0.5
    )
).to(device)

optimizer_2 = torch.optim.RMSprop(
    model_2.fc.parameters(),
    lr=lr_head,
    weight_decay=1e-3
)

scheduler_2 = torch.optim.lr_scheduler.OneCycleLR(
    optimizer_2,
    max_lr = lr_head,
    epochs = 20,
    steps_per_epoch = len(train_loader_2)
)

criterion = nn.CrossEntropyLoss()

for epoch in range(20):
    model_2.train()
    train_loss = 0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader_2:

        images = images.to(device)
        labels = labels.to(device)

        images = train_aug_2(images)

        mask = (labels == 35) | (labels == 36)

        if mask.any():
            images[mask] = targeted_aug_2(images[mask])

        optimizer_2.zero_grad()

        outputs = model_2(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer_2.step()

        scheduler_2.step()

        train_loss += loss.item()

        _, predicted = outputs.max(1)

        train_correct += (predicted == labels).sum().item()

        train_total += labels.size(0)

    train_acc = 100 * train_correct / train_total

    model_2.eval()

    valid_loss = 0
    valid_correct = 0
    valid_total = 0

    with torch.no_grad():

        for images, labels in valid_loader_2:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model_2(images)

            loss = criterion(outputs, labels)

            valid_loss += loss.item()

            _, predicted = outputs.max(1)

            valid_correct += (predicted == labels).sum().item()

            valid_total += labels.size(0)

    valid_acc = 100 * valid_correct / valid_total

    print(
        f"Epoch {epoch+1}/20 | "
        f"Train Acc: {train_acc:.2f}% | "
        f"Valid Acc: {valid_acc:.2f}%"
    )

Epoch 1/20 | Train Acc: 59.46% | Valid Acc: 86.71%
Epoch 2/20 | Train Acc: 81.53% | Valid Acc: 89.88%
Epoch 3/20 | Train Acc: 81.43% | Valid Acc: 87.38%
Epoch 4/20 | Train Acc: 80.40% | Valid Acc: 89.98%
Epoch 5/20 | Train Acc: 80.95% | Valid Acc: 90.94%
Epoch 6/20 | Train Acc: 82.30% | Valid Acc: 91.91%
Epoch 7/20 | Train Acc: 82.78% | Valid Acc: 83.62%
Epoch 8/20 | Train Acc: 85.38% | Valid Acc: 89.79%
Epoch 9/20 | Train Acc: 84.55% | Valid Acc: 92.39%
Epoch 10/20 | Train Acc: 85.38% | Valid Acc: 93.06%
Epoch 11/20 | Train Acc: 86.99% | Valid Acc: 92.58%
Epoch 12/20 | Train Acc: 85.45% | Valid Acc: 93.93%
Epoch 13/20 | Train Acc: 86.73% | Valid Acc: 95.66%
Epoch 14/20 | Train Acc: 89.56% | Valid Acc: 91.91%
Epoch 15/20 | Train Acc: 89.14% | Valid Acc: 95.57%
Epoch 16/20 | Train Acc: 90.94% | Valid Acc: 96.63%
Epoch 17/20 | Train Acc: 92.32% | Valid Acc: 95.86%
Epoch 18/20 | Train Acc: 93.35% | Valid Acc: 97.30%
Epoch 19/20 | Train Acc: 94.03% | Valid Acc: 97.59%
Epoch 20/20 | Train A

In [17]:
transform_3 = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

dataset_3 = dset(image_paths = image_paths, transform = transform_3)

ts = int(0.75*len(dataset_3))
vs = len(dataset_3) - ts
generator = torch.Generator().manual_seed(42)
train_dataset, valid_dataset = random_split(dataset_3, [ts, vs], generator)

train_loader_3 = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True
)

valid_loader_3 = DataLoader(
    valid_dataset,
    batch_size=16,
    shuffle=False
)

model_3 = torchvision.models.resnet34(weights="DEFAULT")

model_3.fc = nn.Sequential(
    nn.Dropout(p=0.5),
    nn.Linear(
        model_3.fc.in_features,
        num_classes
    )
)

model_3 = model_3.to(device)

for param in model_3.parameters():
    param.requires_grad = False

for param in model_3.fc.parameters():
    param.requires_grad = True

train_aug_3 = K.AugmentationSequential(
    K.ColorJiggle(
        contrast = 0.2,
        p = 0.5
    ),
    K.RandomPlanckianJitter(
        mode = "CIED",
        p = 0.5
    )
).to(device)

targeted_aug_3 = K.AugmentationSequential(
    K.RandomRotation(
        degrees = 10,
        p = 0.5
    ),
    K.RandomAffine(
        degrees = 0,
        scale = (0.9, 1.1),
        p = 0.5
    ),
    K.RandomPerspective(
        distortion_scale = 0.2,
        p = 0.5
    ),
    K.ColorJiggle(
        brightness = 0.2,
        contrast = 0.2,
        p = 0.5
    )
).to(device)

optimizer_3 = torch.optim.RMSprop(
    model_3.fc.parameters(),
    lr=lr_head,
    weight_decay=1e-3
)

scheduler_3 = torch.optim.lr_scheduler.OneCycleLR(
    optimizer_3,
    max_lr = lr_head,
    epochs = 20,
    steps_per_epoch = len(train_loader_3)
)

criterion = nn.CrossEntropyLoss()

for epoch in range(20):
    model_3.train()
    train_loss = 0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader_3:

        images = images.to(device)
        labels = labels.to(device)

        images = train_aug_3(images)

        mask = (labels == 35) | (labels == 36)

        if mask.any():
            images[mask] = targeted_aug_3(images[mask])

        optimizer_3.zero_grad()

        outputs = model_3(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer_3.step()

        scheduler_3.step()

        train_loss += loss.item()

        _, predicted = outputs.max(1)

        train_correct += (predicted == labels).sum().item()

        train_total += labels.size(0)

    train_acc = 100 * train_correct / train_total

    model_3.eval()

    valid_loss = 0
    valid_correct = 0
    valid_total = 0

    with torch.no_grad():

        for images, labels in valid_loader_3:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model_3(images)

            loss = criterion(outputs, labels)

            valid_loss += loss.item()

            _, predicted = outputs.max(1)

            valid_correct += (predicted == labels).sum().item()

            valid_total += labels.size(0)

    valid_acc = 100 * valid_correct / valid_total

    print(
        f"Epoch {epoch+1}/20 | "
        f"Train Acc: {train_acc:.2f}% | "
        f"Valid Acc: {valid_acc:.2f}%"
    )

Epoch 1/20 | Train Acc: 50.53% | Valid Acc: 82.47%
Epoch 2/20 | Train Acc: 74.33% | Valid Acc: 86.13%
Epoch 3/20 | Train Acc: 73.08% | Valid Acc: 86.03%
Epoch 4/20 | Train Acc: 72.18% | Valid Acc: 88.15%
Epoch 5/20 | Train Acc: 74.49% | Valid Acc: 91.14%
Epoch 6/20 | Train Acc: 73.31% | Valid Acc: 88.82%
Epoch 7/20 | Train Acc: 75.46% | Valid Acc: 90.85%
Epoch 8/20 | Train Acc: 75.49% | Valid Acc: 88.44%
Epoch 9/20 | Train Acc: 77.93% | Valid Acc: 90.85%
Epoch 10/20 | Train Acc: 76.58% | Valid Acc: 90.46%
Epoch 11/20 | Train Acc: 77.16% | Valid Acc: 91.71%
Epoch 12/20 | Train Acc: 78.03% | Valid Acc: 93.55%
Epoch 13/20 | Train Acc: 79.89% | Valid Acc: 91.33%
Epoch 14/20 | Train Acc: 79.31% | Valid Acc: 91.71%
Epoch 15/20 | Train Acc: 81.11% | Valid Acc: 94.51%
Epoch 16/20 | Train Acc: 84.20% | Valid Acc: 94.80%
Epoch 17/20 | Train Acc: 84.65% | Valid Acc: 94.12%
Epoch 18/20 | Train Acc: 85.67% | Valid Acc: 96.53%
Epoch 19/20 | Train Acc: 87.12% | Valid Acc: 95.76%
Epoch 20/20 | Train A

I will be testing Stochastic depth in other file as we will require unfreezing the backbone